In [5]:
import numpy as np
import pandas as pd
from pypfopt import black_litterman, risk_models
from pypfopt import BlackLittermanModel, EfficientFrontier

# 1. Données simulées
tickers = ["AAPL", "MSFT", "GOOG", "AMZN", "TSLA"]
cov_matrix = pd.DataFrame(
    np.diag([0.05, 0.04, 0.03, 0.05, 0.1]), 
    index=tickers, columns=tickers
)

# 2. Capitalisation boursière (pour calculer les poids du marché)
market_caps = {
    "AAPL": 2800, "MSFT": 2500, "GOOG": 1600, "AMZN": 1300, "TSLA": 800
}

# 3. Calcul du rendement d'équilibre (Implying returns from market cap)
# Le paramètre delta représente l'aversion au risque (généralement autour de 2.5)
delta = 2.5
prior_returns = black_litterman.market_implied_prior_returns(market_caps, delta, cov_matrix)

# 4. Injection des "VUES" du gérant
# On pense que TSLA va faire +15% et AAPL +5%
viewdict = {"TSLA": 0.15, "AAPL": 0.05}

# 5. Création du modèle Black-Litterman
bl = BlackLittermanModel(cov_matrix, pi=prior_returns, absolute_views=viewdict)

# Calcul des rendements espérés ajustés (Posterior Returns)
rets_bl = bl.bl_returns()

# 6. Optimisation finale (Frontière efficiente)
ef = EfficientFrontier(rets_bl, cov_matrix)
weights = ef.max_sharpe() # On cherche le meilleur Ratio de Sharpe
cleaned_weights = ef.clean_weights()

print("Poids optimisés par Black-Litterman :")
print(cleaned_weights)

Poids optimisés par Black-Litterman :
OrderedDict([('AAPL', 0.2735), ('MSFT', 0.21368), ('GOOG', 0.13675), ('AMZN', 0.11111), ('TSLA', 0.26496)])
